# Stable Diffusion 1.5 LoRA trainer (1-click)

Every setting has a default, so add your images and press run. If no image has a caption file, the WD tagger tags them.
The LoRA files go to `MyDrive/lora/<project>/output` on Google Drive.

Any other sd-scripts setting goes in `config_overrides` in the cell's code. The
[advanced notebook](https://github.com/Linaqruf/kohya-trainer/blob/main/notebooks/sd15_lora_advanced.ipynb)
runs the same steps one cell at a time, with the tagger settings and tag editing.

This notebook asks Colab for an L4 GPU, which needs a [paid Colab plan](https://colab.research.google.com/signup). To use another GPU, open Runtime > Change runtime type. A T4 is slower.

*Built by `scripts/build_notebooks.py` in the repo. Your changes stay in your copy.*

In [ ]:
#@title Setup (installs the trainer in 2 to 4 minutes)
import os, subprocess, sys

ROOT = "/content/kohya-trainer"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/Linaqruf/kohya-trainer", ROOT], check=True)
    # Fetches the pinned commit, even after upstream moves on.
    subprocess.run(["git", "-C", ROOT, "submodule", "update", "--init", "--depth=1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ROOT], check=True)
# Python loads an editable install only at start, so add src for this session.
sys.path.insert(0, os.path.join(ROOT, "src"))

from kohya_trainer import environment
environment.install()
loaded = environment.load_tokens()
print("runtime:", environment.runtime_report())
print("tokens loaded from Colab Secrets:", ", ".join(loaded) or "none (add HF_TOKEN for gated or private Hugging Face repos, WANDB_API_KEY for log_to_wandb, CIVITAI_TOKEN for Civitai downloads that need a login)")

In [ ]:
#@title Train
#@markdown **Project**
project_name = "my-lora" #@param {type:"string"}
mount_google_drive = True #@param {type:"boolean"}
#@markdown After a disconnect, tick this to continue from the last save. Keep the model and network fields the same.
resume_from_state = False #@param {type:"boolean"}
#@markdown ---
#@markdown **Model**
base_model = "anylora" #@param ["anylora", "counterfeit-v3.0", "sd-1.5"]
custom_base = "" #@param {type:"string", placeholder:"a Hugging Face or Civitai link, used instead of the list"}
#@markdown ---
#@markdown **Dataset**. A folder, a .zip, a link to a .zip, or a Hugging Face dataset repo (user/name). A subfolder named `10_mychar` trains with 10 repeats. A caption file is a .txt with the same name as its image. `auto_tag` tags the images only if none has one.
dataset = "/content/drive/MyDrive/dataset.zip" #@param {type:"string"}
auto_tag = True #@param {type:"boolean"}
trigger_word = "" #@param {type:"string", placeholder:"a unique word such as mychar, added first to every caption"}
#@markdown After the tagger runs, this removes hair and eye tags found in most captions, so the trigger word learns the look. It works only with a trigger word.
prune_appearance_tags = True #@param {type:"boolean"}
resolution = 512 #@param {type:"integer"}
#@markdown 0 picks the repeats for about 300 images per epoch.
num_repeats = 0 #@param {type:"integer"}
#@markdown ---
#@markdown **Training**
#@markdown The learning rate, the 3 restarts and about 1,500 steps go together. If you change one, it can overtrain.
max_train_epochs = 10 #@param {type:"integer"}
#@markdown 0 uses the epoch count.
max_train_steps = 0 #@param {type:"integer"}
train_batch_size = 2 #@param {type:"integer"}
#@markdown Type any sd-scripts optimizer, or one from an installed package as package.Name, such as pytorch_optimizer.CAME. CAME usually runs at 1e-4 to 2e-4. The optimizer defaults for Prodigy and ProdigyScheduleFree set your learning rate to 1.0.
optimizer = "AdamW8bit" #@param ["AdamW8bit", "AdamW", "Prodigy", "ProdigyScheduleFree", "CAME", "AdaFactor"] {allow-input: true}
learning_rate = 0.0005 #@param {type:"number"}
text_encoder_lr = 0.0001 #@param {type:"number"}
#@markdown Pick one or type another sd-scripts name. Schedule-free optimizers ignore the scheduler and warmup. With Prodigy, cosine or constant_with_warmup are the usual picks.
lr_scheduler = "cosine_with_restarts" #@param ["constant", "constant_with_warmup", "cosine", "cosine_with_restarts", "linear"] {allow-input: true}
#@markdown Warmup is a share of all steps (0.05 is 5%) or a step count. Only cosine_with_restarts uses the cycles.
lr_warmup_steps = 0.05 #@param {type:"number"}
lr_scheduler_num_cycles = 3 #@param {type:"integer"}
#@markdown Min-SNR gamma: 5 is usual, 0 turns it off.
min_snr_gamma = 5 #@param {type:"number"}
#@markdown ---
#@markdown **Network**. The defaults set alpha to half of dim, so change them together.
network_type = "lora" #@param ["lora", "locon"]
network_dim = 32 #@param {type:"integer"}
network_alpha = 16 #@param {type:"number"}
#@markdown ---
#@markdown **Output**
#@markdown 0 saves only the final LoRA file.
save_every_n_epochs = 1 #@param {type:"integer"}
sample_prompt = "" #@param {type:"string", placeholder:"a prompt, or a .txt file with one per line (empty for the built-in one)"}
upload_to_hf = "" #@param {type:"string", placeholder:"username/repo for each saved epoch (needs a write HF_TOKEN)"}
#@markdown Needs the `WANDB_API_KEY` secret.
log_to_wandb = False #@param {type:"boolean"}

# Other sd-scripts settings go in config_overrides, dataset settings in dataset_overrides. Both win over the form and the preset.
config_overrides = {}
dataset_overrides = {}

from kohya_trainer import environment, train
train.check_project_name(project_name)
if mount_google_drive:
    environment.mount_drive()
plan = train.quick_train(
    model="sd15",
    base=custom_base or base_model,
    dataset=dataset,
    project_name=project_name,
    trigger_word=trigger_word,
    auto_tag=auto_tag,
    prune_appearance_tags=prune_appearance_tags,
    resume=resume_from_state,
    upload_repo=upload_to_hf,
    log_to_wandb=log_to_wandb,
    network_type=network_type,
    network_dim=network_dim,
    network_alpha=network_alpha,
    resolution=resolution,
    train_batch_size=train_batch_size,
    num_repeats=num_repeats or None,
    sample_prompt=sample_prompt or None,
    optimizer=optimizer,
    form_values={
        "learning_rate": learning_rate,
        "text_encoder_lr": text_encoder_lr,
        "max_train_epochs": max_train_epochs,
        "max_train_steps": max_train_steps,
        "lr_scheduler": lr_scheduler,
        "lr_warmup_steps": lr_warmup_steps,
        "lr_scheduler_num_cycles": lr_scheduler_num_cycles,
        "min_snr_gamma": min_snr_gamma,
        "save_every_n_epochs": save_every_n_epochs,
    },
    config_overrides=config_overrides,
    dataset_overrides=dataset_overrides,
)

### After training

- The LoRA files are in `MyDrive/lora/<project_name>/output/`, and the sample images in
  `output/sample/`. Without Drive, they are in `/content/lora/<project_name>/output/`, which
  Colab deletes when the session ends.
- With Drive mounted and `save_every_n_epochs` at 1 (the default), every finished epoch is already
  saved, so a run that Colab stops keeps them.
- A session keeps running and using compute units after the page disconnects. If the page
  comes back on a new session, stop the old one in Runtime > Manage sessions. When you are
  done, choose Runtime > Disconnect and delete runtime.
- If every image shows one outfit, a prompt for that outfit can bring the subject in without the trigger word. If that is a problem, lower the LoRA weight.
- Tags are shuffled, and `keep_tokens = 1` keeps your trigger word first.